# FERRUS FORGE — forge_main.ipynb
## Fregate 00 : Conversion d'avatars bruts en .blend

```
IN/  avatar_P*.glb / avatar_P*.obj / avatar_P*.fbx
         ↓
   forge_convert.py (Blender headless)
         ↓
OUT/ avatar_P*.blend  +  rapport_forge.json
         ↓
FERRUS CORPUS IN_AVATAR/
```

**POUR L'EMPEROR. POUR LA FLOTTE FERRUS.**

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 0 — GIT SYNC
# Monte le Drive + clone/pull le repo + installe le codebase
# A executer EN PREMIER a chaque session Colab
# ═══════════════════════════════════════════════════════════

import os, shutil, subprocess
from google.colab import drive

# ── 0.1 Monter Google Drive ──────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ── 0.2 CONFIGURER ICI ───────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/FLOTTE-FERRUS'
GITHUB_REPO = 'https://github.com/kioka8877-ux/FLOTTE-FERRUS.git'
CLONE_DIR   = '/content/FLOTTE-FERRUS'

# ── 0.3 Clone ou Pull ────────────────────────────────────
if os.path.isdir(os.path.join(CLONE_DIR, '.git')):
    print('[GIT SYNC] Repo deja clone — git pull...')
    proc = subprocess.run(
        ['git', '-C', CLONE_DIR, 'pull', '--rebase'],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Deja a jour.')
else:
    print('[GIT SYNC] Clone du repo...')
    proc = subprocess.run(
        ['git', 'clone', '--depth=1', GITHUB_REPO, CLONE_DIR],
        capture_output=True, text=True
    )
    print(proc.stdout.strip() or 'Clone OK.')
    if proc.returncode != 0:
        print('[GIT SYNC] ERREUR :', proc.stderr[-500:])
        raise RuntimeError('git clone echoue')

# ── 0.4 Installer le codebase sur Drive ──────────────────
SRC_CODEBASE  = os.path.join(CLONE_DIR, '00_FERRUS_FORGE', 'codebase')
DEST_CODEBASE = os.path.join(DRIVE_ROOT, '00_FERRUS_FORGE', 'codebase')
os.makedirs(DEST_CODEBASE, exist_ok=True)

for fname in ['forge_convert.py']:
    src  = os.path.join(SRC_CODEBASE, fname)
    dest = os.path.join(DEST_CODEBASE, fname)
    shutil.copy2(src, dest)
    print(f'[GIT SYNC] Copie : {fname} → Drive/00_FERRUS_FORGE/codebase/')

# ── 0.5 Creer les dossiers requis sur Drive ──────────────
for folder in ['IN', 'OUT']:
    path = os.path.join(DRIVE_ROOT, '00_FERRUS_FORGE', folder)
    os.makedirs(path, exist_ok=True)

# ── 0.6 Bilan ────────────────────────────────────────────
print()
print('[GIT SYNC] ══════════════════════════════════════')
print('[GIT SYNC] Codebase synchronise depuis GitHub')
print(f'[GIT SYNC] DRIVE_ROOT : {DRIVE_ROOT}')
for folder in ['IN', 'OUT', 'codebase']:
    path = os.path.join(DRIVE_ROOT, '00_FERRUS_FORGE', folder)
    print(f'  {"OK" if os.path.isdir(path) else "MANQUANT"}  {path}')
print('[GIT SYNC] Passer a la cellule SETUP ▶')
print('[GIT SYNC] ══════════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 1 — SETUP
# Detecte Blender, copie en local si besoin, prepare les chemins
# ═══════════════════════════════════════════════════════════

import os, glob, subprocess, json, shutil, stat

# ── 1.1 Chemins ──────────────────────────────────────────
FORGE_ROOT    = os.path.join(DRIVE_ROOT, '00_FERRUS_FORGE')
IN_DIR        = os.path.join(FORGE_ROOT, 'IN')
OUT_DIR       = os.path.join(FORGE_ROOT, 'OUT')
CODEBASE_DIR  = os.path.join(FORGE_ROOT, 'codebase')
CONVERT_SCRIPT = os.path.join(CODEBASE_DIR, 'forge_convert.py')

os.makedirs(OUT_DIR, exist_ok=True)

# ── 1.2 Detecter Blender sur Drive ───────────────────────
def _find_blender_on_drive():
    """Cherche le binaire blender sur Drive (chemins connus + scan cible)."""
    candidates = [
        '/content/drive/MyDrive/tools/blender-4.2/blender',
        '/content/drive/MyDrive/tools/blender-4.1/blender',
        '/content/drive/MyDrive/tools/blender-4.0/blender',
        '/content/drive/MyDrive/EXODUS_AI_MODELS/blender-4.0.0-linux-x64/blender',
        '/content/drive/MyDrive/EXODUS_AI_MODELS/BLENDER/blender-4.0.0-linux-x64/blender',
    ]
    for p in candidates:
        if os.path.isfile(p):
            return p
    # Scan cible sur dossiers petits connus
    for root in ['/content/drive/MyDrive/tools', '/content/drive/MyDrive/blender',
                 '/content/drive/MyDrive/OUTILS', '/content/drive/MyDrive/BLENDER']:
        if not os.path.isdir(root):
            continue
        for dirpath, _, filenames in os.walk(root):
            if 'blender' in filenames:
                return os.path.join(dirpath, 'blender')
    return None

# ── 1.3 Installer Blender en local (fix permissions Drive FUSE) ──
def _install_blender_local(drive_bin):
    """Copie le dossier Blender de Drive vers /content/ et fixe les permissions."""
    blender_dir_drive = os.path.dirname(drive_bin)
    blender_dir_local = os.path.join('/content', os.path.basename(blender_dir_drive))
    local_bin         = os.path.join(blender_dir_local, 'blender')

    if os.path.isfile(local_bin) and os.access(local_bin, os.X_OK):
        return local_bin  # Deja installe

    print(f'[SETUP] Copie Blender en local (Drive FUSE ne conserve pas +x)...')
    if os.path.isdir(blender_dir_local):
        shutil.rmtree(blender_dir_local)
    shutil.copytree(blender_dir_drive, blender_dir_local)

    # Chmod +x sur tous les binaires du dossier
    for dirpath, _, filenames in os.walk(blender_dir_local):
        for fname in filenames:
            fpath = os.path.join(dirpath, fname)
            try:
                st = os.stat(fpath)
                os.chmod(fpath, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
            except Exception:
                pass
    print(f'[SETUP] Blender installe : {local_bin}')
    return local_bin

# ── 1.4 Chemins locaux prioritaires (deja installes) ─────
LOCAL_CANDIDATES = [
    '/content/blender-4.2/blender',
    '/content/blender-4.1/blender',
    '/content/blender-4.0/blender',
    '/content/blender/blender',
    '/usr/local/blender/blender',
]
BLENDER_BIN = next((p for p in LOCAL_CANDIDATES if os.path.isfile(p) and os.access(p, os.X_OK)), None)

if not BLENDER_BIN:
    drive_bin = _find_blender_on_drive()
    if drive_bin:
        BLENDER_BIN = _install_blender_local(drive_bin)
    else:
        print('[SETUP] ERREUR — Blender introuvable sur Drive ni en local.')
        print('[SETUP] Placer Blender dans Drive/tools/blender-X.Y/ et relancer.')

if BLENDER_BIN:
    print(f'[SETUP] Blender : {BLENDER_BIN}')

print(f'[SETUP] IN     : {IN_DIR}')
print(f'[SETUP] OUT    : {OUT_DIR}')
print(f'[SETUP] Script : {"OK" if os.path.isfile(CONVERT_SCRIPT) else "MANQUANT"}')
print('[SETUP] OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 2 — CONFIG
# Liste les avatars dans IN/ et valide le nommage P*
# ═══════════════════════════════════════════════════════════

SUPPORTED_EXTS = ('.glb', '.gltf', '.obj', '.fbx')

# ── 2.1 Scanner IN/ ──────────────────────────────────────
all_files  = []
skip_files = []

for f in sorted(os.listdir(IN_DIR)):
    fpath = os.path.join(IN_DIR, f)
    if not os.path.isfile(fpath):
        continue
    ext = os.path.splitext(f)[1].lower()
    if ext in SUPPORTED_EXTS:
        all_files.append(fpath)
    elif f != '.gitkeep':
        skip_files.append(f)

if not all_files:
    print('[CONFIG] ERREUR — Aucun avatar supporte dans IN/')
    print(f'[CONFIG] Formats acceptes : {SUPPORTED_EXTS}')
    print(f'[CONFIG] Deposer les avatars dans : {IN_DIR}')
else:
    print(f'[CONFIG] {len(all_files)} avatar(s) trouve(s) dans IN/ :')
    print(f'  {"FICHIER":<35} {"FORMAT":<8} {"TAILLE"}')
    print('  ' + '─' * 55)
    for f in all_files:
        ext     = os.path.splitext(f)[1].lower().lstrip('.')
        size_ko = os.path.getsize(f) // 1024
        print(f'  {os.path.basename(f):<35} {ext:<8} {size_ko} Ko')

if skip_files:
    print()
    print('[CONFIG] Fichiers ignores (format non supporte) :')
    for f in skip_files:
        print(f'  {f}')

# ── 2.2 Validation nommage miroir ────────────────────────
print()
print('[CONFIG] Validation nommage miroir (avatar_P*.ext) :')
naming_ok = True
for f in all_files:
    stem = os.path.splitext(os.path.basename(f))[0]  # ex: avatar_P1
    if not stem.startswith('avatar_P'):
        print(f'  WARNING — Nommage non conforme : {os.path.basename(f)}')
        print(f'           Renommer en : avatar_P*.ext pour la compatibilite CORPUS')
        naming_ok = False
    else:
        print(f'  OK  {os.path.basename(f)}')

# ── 2.3 Bilan ────────────────────────────────────────────
ready = bool(all_files and os.path.isfile(CONVERT_SCRIPT) and BLENDER_BIN)
print()
print('[CONFIG] BILAN ──────────────────────────────────')
print(f'  Avatars dispos : {len(all_files)}')
print(f'  Nommage miroir : {"OK" if naming_ok else "WARNINGS (voir ci-dessus)"}')
print(f'  Script         : {"OK" if os.path.isfile(CONVERT_SCRIPT) else "MANQUANT"}')
print(f'  Blender        : {"OK" if BLENDER_BIN else "MANQUANT"}')
print(f'  GO             : {"OUI" if ready else "NON — corriger les points ci-dessus"}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 3 — CONVERSION
# Boucle sur N avatars, appelle forge_convert.py via Blender
# ═══════════════════════════════════════════════════════════

import datetime

if not ready:
    print('[CONVERSION] Conditions non remplies — executer CONFIG dabord')
    raise SystemExit

rapport_global = {
    'generated_at':   datetime.datetime.now().isoformat(),
    'blender_bin':    BLENDER_BIN,
    'total_avatars':  len(all_files),
    'avatars':        []
}

for input_path in all_files:
    basename  = os.path.basename(input_path)
    stem      = os.path.splitext(basename)[0]   # ex: avatar_P1
    actor_id  = stem.replace('avatar_', '')      # ex: P1
    out_blend = os.path.join(OUT_DIR, f'{stem}.blend')
    report_tmp = os.path.join(OUT_DIR, f'_tmp_report_{actor_id}.json')

    print(f'[CONVERSION] ─────────────────────────────────')
    print(f'[CONVERSION] Acteur  : {actor_id}')
    print(f'[CONVERSION] Source  : {basename}')
    print(f'[CONVERSION] Sortie  : {os.path.basename(out_blend)}')

    cmd = [
        BLENDER_BIN, '--background',
        '--python', CONVERT_SCRIPT,
        '--',
        '--input',        input_path,
        '--output-blend', out_blend,
        '--report-json',  report_tmp,
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=180)

    # Afficher les logs FORGE
    forge_lines = [l for l in proc.stdout.splitlines() if '[FORGE]' in l]
    for line in forge_lines:
        print(line)

    if proc.returncode != 0:
        print(f'[CONVERSION] ERREUR Blender (code {proc.returncode})')
        error_lines = [l for l in proc.stderr.splitlines() if 'Error' in l or 'error' in l]
        for line in error_lines[-5:]:
            print(f'  {line}')
        rapport_global['avatars'].append({
            'id':          actor_id,
            'source_file': basename,
            'status':      'ERREUR',
            'return_code': proc.returncode
        })
        continue

    # Lire le rapport individuel
    actor_report = {'id': actor_id, 'status': 'OK'}
    if os.path.isfile(report_tmp):
        with open(report_tmp) as f:
            actor_data = json.load(f)
        actor_report.update(actor_data)
        os.remove(report_tmp)

    rapport_global['avatars'].append(actor_report)
    print(f'[CONVERSION] Acteur {actor_id} OK')

# Ecrire le rapport global
rapport_path = os.path.join(OUT_DIR, 'rapport_forge.json')
with open(rapport_path, 'w') as f:
    json.dump(rapport_global, f, indent=2)

ok_count  = sum(1 for a in rapport_global['avatars'] if a.get('status') == 'OK')
err_count = len(rapport_global['avatars']) - ok_count

print()
print('[CONVERSION] ═══════════════════════════════════')
print(f'[CONVERSION] TERMINE — {ok_count}/{len(all_files)} avatar(s) convertis')
if err_count:
    print(f'[CONVERSION] ATTENTION — {err_count} erreur(s)')
print(f'[CONVERSION] Rapport : {rapport_path}')
print('[CONVERSION] ═══════════════════════════════════')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELLULE 4 — RAPPORT
# Affiche rapport_forge.json + resume des warnings
# ═══════════════════════════════════════════════════════════

rapport_path = os.path.join(OUT_DIR, 'rapport_forge.json')

if not os.path.isfile(rapport_path):
    print('[RAPPORT] rapport_forge.json introuvable — executer CONVERSION dabord')
else:
    with open(rapport_path) as f:
        r = json.load(f)

    print(f'[RAPPORT] Genere le     : {r["generated_at"]}')
    print(f'[RAPPORT] Total avatars : {r["total_avatars"]}')
    print()
    print(f'{"ACTEUR":<10} {"SOURCE":<30} {"FORMAT":<8} {"ARMATURE":<10} {"R15":<6} {"BLEND (Ko)":<12} {"STATUT"}')
    print('─' * 85)
    for a in r['avatars']:
        if a.get('status') == 'OK':
            print(
                f'{a["id"]:<10} '
                f'{a.get("source_file", "?"):<30} '
                f'{a.get("format_detected", "?"):<8} '
                f'{str(a.get("armature_found", "?")):<10} '
                f'{str(a.get("r15_bones_found", "?")):<6} '
                f'{a.get("output_blend_size_ko", "?"):<12} '
                f'{a["status"]}'
            )
        else:
            print(f'{a["id"]:<10} {a.get("source_file", "?"):<30} {"─":<8} {"─":<10} {"─":<6} {"─":<12} ERREUR')

    # Warnings
    all_warnings = [
        (a['id'], w)
        for a in r['avatars']
        for w in a.get('warnings', [])
    ]
    if all_warnings:
        print()
        print('[RAPPORT] WARNINGS ──────────────────────────────')
        for actor_id, msg in all_warnings:
            print(f'  [{actor_id}] {msg}')

    # Instructions suite
    ok_actors = [a for a in r['avatars'] if a.get('status') == 'OK']
    if ok_actors:
        print()
        print('[RAPPORT] SUITE — Copier les .blend vers FERRUS CORPUS :')
        corpus_avatar_dir = OUT_DIR.replace('00_FERRUS_FORGE/OUT', '02_FERRUS_CORPUS/IN_AVATAR')
        print(f'  Source      : {OUT_DIR}')
        print(f'  Destination : {corpus_avatar_dir}')
        print(f'  Fichiers    : {" | ".join(a.get("output_blend", "") for a in ok_actors)}')